In [48]:
import duckdb

con = duckdb.connect("../data/nyc.duckdb", read_only=True)
con.sql("LOAD spatial")

con.sql("SHOW TABLES").df()

,name
0,raw_nta
1,raw_pluto
2,raw_streeteasy
3,stg_pluto__lots


In [49]:
import pandas as pd
pd.set_option("display.max_rows", 200)

In [53]:
con.sql("""
select count(*) as n_rental_lots
from raw_pluto
where (bldgclass like 'C%' or bldgclass like 'D%')
  and try_cast(unitsres as double) >= 3
  and bldgclass not in ('C6', 'C8', 'D0', 'D4', 'CC', 'CM', 'DC')
""").df()

,n_rental_lots
0,142940


In [36]:
con.sql("""
select count(*) as n_failed_casts
from raw_pluto
where unitsres is not null
  and try_cast(unitsres as double) is null
""").df()

,n_failed_casts
0,0


In [37]:
con.sql("""
with rental as (
    select *
    from raw_pluto
    where (bldgclass like 'C%' or bldgclass like 'D%')
      and try_cast(unitsres as double) >= 3
      and bldgclass not in ('C6', 'C8', 'D0', 'D4')
)
select
    count(*)                                   as n_lots,
    count(*) - count(bbl)                      as null_bbl,
    count(*) - count(bldgclass)                as null_bldgclass,
    count(*) - count(unitsres)                 as null_unitsres,
    count(*) - count(numfloors)                as null_numfloors,
    count(*) - count(yearbuilt)                as null_yearbuilt,
    count(*) - count(latitude)                 as null_latitude,
    count(*) - count(longitude)                as null_longitude,
    count_if(try_cast(numfloors as double) = 0) as zero_numfloors,
    count_if(try_cast(yearbuilt as double) = 0) as zero_yearbuilt
from rental
""").df()

,n_lots,null_bbl,null_bldgclass,null_unitsres,null_numfloors,null_yearbuilt,null_latitude,null_longitude,zero_numfloors,zero_yearbuilt
0,142944,0,0,0,276,0,75,75,0.0,59.0


In [38]:
con.sql("""
with long as (
    unpivot raw_streeteasy
    on columns(* exclude (areaName, Borough, areaType))
    into name month value rent
),
neighborhood_months as (
    select
        areaName,
        Borough,
        strptime(month, '%Y-%m')::date as month_date
    from long
    where areaType = 'neighborhood'
      and try_cast(rent as double) is not null
)
select
    areaName,
    Borough,
    min(month_date)  as first_month,
    max(month_date)  as last_month,
    count(*)         as n_months,
    datediff('month', min(month_date), max(month_date)) + 1 - count(*) as n_gap_months
from neighborhood_months
group by areaName, Borough
order by first_month desc, areaName
""").df()

,areaName,Borough,first_month,last_month,n_months,n_gap_months
0,Cambria Heights,Queens,2015-04-01,2026-10-01,78,61
1,Queens Village,Queens,2013-12-01,2026-10-01,154,1
2,Hollis,Queens,2013-08-01,2026-10-01,150,9
3,Laurelton,Queens,2013-08-01,2026-10-01,120,39
4,Seagate,Brooklyn,2013-02-01,2026-10-01,108,57
5,Co-op City,Bronx,2012-11-01,2025-05-01,27,124
6,City Island,Bronx,2012-09-01,2026-08-01,116,52
7,Old Mill Basin,Brooklyn,2012-09-01,2026-10-01,115,55
8,South Jamaica,Queens,2012-05-01,2026-10-01,145,29
9,Mill Basin,Brooklyn,2012-03-01,2026-09-01,140,35


In [39]:
con.sql("""
select
    bldgclass,
    count(*) as n_lots
from raw_pluto
where bldgclass like 'C%' or bldgclass like 'D%'
group by bldgclass
order by bldgclass
""").df()

,bldgclass,n_lots
0,C0,71511
1,C1,16725
2,C2,13568
3,C3,15914
4,C4,2862
5,C5,2377
6,C6,3345
7,C7,7722
8,C8,16
9,C9,399


## Questions raised

**PLUTO (142,944 rental lots in scope)**
- No NTA column in PLUTO 26v3, so lots need a point-in-polygon join to the 2020 NTAs (Day 3).
- 75 rental lots have no latitude/longitude and can't be placed in an NTA.
- 59 rental lots have yearbuilt = 0 (unknown). Staging sets these to NULL.
- 276 rental lots have no numfloors.
- `ownername` is in the raw table. Staging must not select it (no owner names).

**StreetEasy (176 neighborhoods)**
- All 176 have rent history; 82 have all 202 months with no gaps.
- 43 are missing 24+ months within their own range, mostly in the Bronx, Queens and south Brooklyn.
- 11 don't have Oct 2026 as their last month; Co-op City, Floral Park and New Hyde Park stopped in 2025.
- Open: does "latest month" mean Oct 2026 for all (NULL where missing), or each neighborhood's own latest?
- Open: how many of the trailing 12 months must have a value before the 12-month median counts?

In [58]:
con.sql("""
select count(*) 
from raw_pluto
where unitsres LIKE '%.%'
""").df()

,count_star()
0,0


In [56]:
con.sql("""
select 
count(*), 
count(bbl), 
count(bldgclass), 
count(unitsres), 
count(numfloors), 
count(yearbuilt), 
count(latitude), 
count(longitude)
from stg_pluto__lots
""").df()

,count_star(),count(bbl),count(bldgclass),count(unitsres),count(numfloors),count(yearbuilt),count(latitude),count(longitude)
0,858312,858312,858004,857904,816103,818160,857127,857127


In [55]:
con.sql("""
select numfloors 
from raw_pluto
group by 1
order by 1
""").df()

,numfloors
0,0.0000000
1,1.0000000
2,1.0100000
3,1.1000000
4,1.1500000
5,1.1700000
6,1.2000000
7,1.2500000
8,1.3100000
9,1.3300000


In [59]:
con.close()